# Module 13: Statistics for Data

**Lesson: Descriptive Statistics, Probability Distributions, Hypothesis Testing, and Correlation**

In this lesson we explore core statistical concepts with hands-on Python examples using real datasets. We focus on how statistics informs machine learning — from feature selection to model assumptions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import fetch_california_housing
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Libraries loaded successfully')

## 1. Descriptive Statistics

We load the Iris dataset and compute central tendency and dispersion measures.

In [ ]:
# Load Iris dataset
iris = sns.load_dataset('iris')
print('Iris dataset shape:', iris.shape)
print(iris.head())

# Descriptive statistics
print('\n--- Descriptive Statistics ---')
for col in ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']:
    data = iris[col]
    print(f'\n{col}:')
    print(f'  Mean: {data.mean():.3f}')
    print(f'  Median: {data.median():.3f}')
    print(f'  Variance: {data.var():.3f}')
    print(f'  Std Dev: {data.std():.3f}')
    print(f'  IQR: {data.quantile(0.75) - data.quantile(0.25):.3f}')
    print(f'  Skewness: {stats.skew(data):.3f}')
    print(f'  Kurtosis: {stats.kurtosis(data):.3f}')

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, col in zip(axes.flat, ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']):
    for species in iris['species'].unique():
        subset = iris[iris['species'] == species][col]
        ax.hist(subset, bins=15, alpha=0.5, label=species)
    ax.set_title(f'{col} Distribution by Species')
    ax.legend()
plt.tight_layout()
plt.show()

## 2. Probability Distributions

We simulate samples from normal, binomial, and Poisson distributions and compare them to real data.

In [ ]:
# Normal distribution
mu, sigma = 5.0, 0.5
normal_samples = np.random.normal(mu, sigma, 10000)

# Binomial distribution (n=10, p=0.5)
binomial_samples = np.random.binomial(n=10, p=0.5, size=10000)

# Poisson distribution (lambda=3)
poisson_samples = np.random.poisson(lam=3.0, size=10000)

# Plot all three
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(normal_samples, bins=50, density=True, alpha=0.7, color='blue')
axes[0].set_title(f'Normal(mu={mu}, sigma={sigma})')
axes[1].hist(binomial_samples, bins=11, density=True, alpha=0.7, color='green')
axes[1].set_title('Binomial(n=10, p=0.5)')
axes[2].hist(poisson_samples, bins=15, density=True, alpha=0.7, color='red')
axes[2].set_title('Poisson(lambda=3)')
plt.tight_layout()
plt.show()

print('Normal - 68% within 1 std:', np.mean(np.abs(normal_samples - mu) < sigma))
print('Normal - 95% within 2 std:', np.mean(np.abs(normal_samples - mu) < 2*sigma))

## 3. Central Limit Theorem

Demonstrate how sample means approach a normal distribution regardless of the population distribution.

In [ ]:
# Population: exponential distribution (very non-normal)
population = np.random.exponential(scale=2.0, size=100000)

# Draw samples of increasing size
sample_sizes = [5, 30, 100]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, n in enumerate(sample_sizes):
    sample_means = [np.mean(np.random.choice(population, n)) for _ in range(1000)]
    axes[i].hist(sample_means, bins=40, density=True, alpha=0.7, color='purple')
    axes[i].set_title(f'Sample Means (n={n})')
    axes[i].axvline(np.mean(sample_means), color='red', linestyle='--', label=f'Mean: {np.mean(sample_means):.3f}')
    axes[i].axvline(np.mean(population), color='green', linestyle='--', label=f'Pop Mean: {np.mean(population):.3f}')
    axes[i].legend()

plt.tight_layout()
plt.show()
print(f'Population mean: {np.mean(population):.3f}, Population std: {np.std(population):.3f}')
print(f'CLT prediction - std error for n=30: {np.std(population)/np.sqrt(30):.3f}')

## 4. Hypothesis Testing

Perform t-tests, chi-square tests, and ANOVA on real data.

In [ ]:
# t-test: compare sepal length between setosa and versicolor
setosa = iris[iris['species'] == 'setosa']['sepal_length']
versicolor = iris[iris['species'] == 'versicolor']['sepal_length']

t_stat, p_value = stats.ttest_ind(setosa, versicolor)
print('=== Two-Sample t-test: Setosa vs Versicolor (Sepal Length) ===')
print(f't-statistic: {t_stat:.4f}')
print(f'p-value: {p_value:.6f}')
print(f'Setosa mean: {setosa.mean():.3f}, Versicolor mean: {versicolor.mean():.3f}')
print('Conclusion: p < 0.05 => significant difference')

# Cohen's d effect size
pooled_std = np.sqrt((setosa.std()**2 + versicolor.std()**2) / 2)
cohens_d = (versicolor.mean() - setosa.mean()) / pooled_std
print(f"Cohen's d: {cohens_d:.3f} (large effect if > 0.8)")

In [ ]:
# ANOVA: compare sepal length across all three species
setosa = iris[iris['species'] == 'setosa']['sepal_length']
versicolor = iris[iris['species'] == 'versicolor']['sepal_length']
virginica = iris[iris['species'] == 'virginica']['sepal_length']

f_stat, p_value = stats.f_oneway(setosa, versicolor, virginica)
print('=== One-Way ANOVA: Sepal Length across 3 Species ===')
print(f'F-statistic: {f_stat:.4f}')
print(f'p-value: {p_value:.10f}')
print('Conclusion: p < 0.05 => at least one species differs significantly')

In [ ]:
# Chi-square test for independence on Titanic data
titanic = sns.load_dataset('titanic')
titanic = titanic.dropna(subset=['sex', 'survived'])
contingency = pd.crosstab(titanic['sex'], titanic['survived'])
print('Contingency Table:')
print(contingency)

chi2, p, dof, expected = stats.chi2_contingency(contingency)
print(f'\nChi-square: {chi2:.4f}')
print(f'p-value: {p:.6f}')
print(f'Degrees of freedom: {dof}')
print('Conclusion: p < 0.05 => sex and survival are not independent')

## 5. Correlation Analysis

Pearson and Spearman correlation on California housing features.

In [ ]:
# Load California Housing
housing = fetch_california_housing(as_frame=True)
df = housing.frame

# Pearson correlation
pearson_corr = df.corr(method='pearson')
spearman_corr = df.corr(method='spearman')

print('=== Pearson Correlation with MedHouseVal ===')
print(pearson_corr['MedHouseVal'].sort_values(ascending=False))

print('\n=== Spearman Correlation with MedHouseVal ===')
print(spearman_corr['MedHouseVal'].sort_values(ascending=False))

# Heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(pearson_corr, annot=True, fmt='.2f', cmap='RdBu_r', 
            vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title('Pearson Correlation')
sns.heatmap(spearman_corr, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title('Spearman Correlation')
plt.tight_layout()
plt.show()

## 6. ML Feature Selection with Statistical Tests

Use ANOVA F-test to select the best features for a classification task on Iris.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, chi2
from sklearn.preprocessing import MinMaxScaler

# Features and target
X = iris.drop('species', axis=1)
y = iris['species']

# ANOVA F-test feature selection
selector = SelectKBest(score_func=f_classif, k=2)
X_selected = selector.fit_transform(X, y)

# Feature scores
scores = pd.DataFrame({
    'feature': X.columns,
    'f_score': selector.scores_,
    'p_value': selector.pvalues_
}).sort_values('f_score', ascending=False)

print('=== Feature Selection with ANOVA F-test ===')
print(scores)
print(f'\nSelected features: {X.columns[selector.get_support()].tolist()}')

# Chi-square requires non-negative features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
chi2_selector = SelectKBest(score_func=chi2, k=2)
chi2_selector.fit_transform(X_scaled, y)
print(f'\nChi2 selected features: {X.columns[chi2_selector.get_support()].tolist()}')

## 7. Bayes Theorem Intuition

A simple medical testing scenario to build intuition for Bayesian reasoning.

In [ ]:
# Bayes theorem: Medical test example
# Disease prevalence: 1% (prior)
# Test sensitivity (true positive rate): 99%
# Test false positive rate: 5%

prior = 0.01       # P(Disease)
sensitivity = 0.99  # P(Positive | Disease)
false_positive = 0.05  # P(Positive | No Disease)

# P(Positive)
p_positive = prior * sensitivity + (1 - prior) * false_positive

# P(Disease | Positive) - posterior
posterior = (prior * sensitivity) / p_positive

print('=== Bayes Theorem: Medical Test ===')
print(f'Prior probability of disease: {prior:.1%}')
print(f'Test sensitivity: {sensitivity:.0%}')
print(f'False positive rate: {false_positive:.0%}')
print(f'Posterior P(Disease|Positive): {posterior:.2%}')
print('\nInterpretation: Even with a positive test, only {:.1f}% chance of having the disease'.format(posterior * 100))
print('This is the base rate fallacy in action!')

## Summary

- **Descriptive stats**: Mean, median, variance, std, IQR summarize data distributions
- **Probability distributions**: Normal, binomial, Poisson model different data types
- **CLT**: Sample means are normally distributed for large n, regardless of population
- **Hypothesis testing**: t-test (2 groups), ANOVA (3+ groups), chi-square (categorical)
- **Correlation**: Pearson (linear), Spearman (monotonic) — always check scatter plots
- **Effect size**: p-values alone are insufficient — Cohen's d measures practical significance
- **Bayes**: Prior + evidence = posterior — fundamental to probabilistic ML
- **ML relevance**: Statistical tests drive feature selection and validate model assumptions